# Semantic Model Optimization Scanner — V2.1.1

This notebook finds semantic-model optimization opportunities and writes the results to the attached Lakehouse. It does **not** change the scanned model and does not claim CU savings; CU improvement is measured later after an approved change.

## Run it

1. Attach the output Lakehouse to this notebook.
2. Enter the IDs in **Member inputs**.
3. Select **Run all**, or call the notebook from a Fabric Pipeline.

| Input | What to enter |
|---|---|
| `workspace_ids` | Required. One or more Workspace IDs separated by commas, semicolons, spaces, or new lines. |
| `model_ids_optional` | Optional. Leave blank to scan every semantic model in the listed workspaces; otherwise enter one or more Model IDs. |
| `requested_by_upn_optional` | Optional requester email for the audit log. |
| `_inlineInstallationEnabled` | System setting for Pipeline package installation. Leave it as `True`. |

Examples:

```text
workspace_ids = "workspace-id-1, workspace-id-2"
model_ids_optional = ""                         # all models in both workspaces
model_ids_optional = "model-id-1, model-id-2"  # selected models only
```

The platform owner configures authentication and safeguards once in the collapsed **Platform setup** cell. For an SPN run, the target workspace owner must add the scanner SPN as Contributor or Member before the scan.


In [ ]:
# Packages are installed for this notebook session; a Fabric Environment is optional.
%pip install -q "semantic-link-sempy==0.14.2" "semantic-link-labs==0.15.2"


In [ ]:
# MEMBER INPUTS — these are the only values auto-populated in a Pipeline Notebook activity.

workspace_ids = ""                 # Required: id1,id2 or one ID per line
model_ids_optional = ""             # Optional: blank = every model in the listed workspaces
requested_by_upn_optional = ""      # Optional: requester email for audit

# Fabric system switch required for the package-install cell during Pipeline runs.
# Leave this value unchanged.
_inlineInstallationEnabled = True
initialize_only = False               # Deployment use only; not exposed by the pipeline


In [ ]:
# PLATFORM SETUP — configured once by the scanner owner; regular members do not edit.

# Identity: user | spn_secret | spn_keyvault
auth_mode = "user"
spn_tenant_id = ""
spn_client_id = ""
spn_object_id = ""                 # Entra enterprise application Object ID
spn_client_secret = ""              # Test only; do not save a real secret here
allow_plaintext_spn_secret = False

# Azure Key Vault settings for production SPN authentication
key_vault_name_or_uri = ""
kv_tenant_id_secret_name = ""       # Optional when spn_tenant_id is set above
kv_client_id_secret_name = ""       # Optional when spn_client_id is set above
kv_client_secret_name = ""          # Required for spn_keyvault

# Standard analysis profile
analysis_profile = "standard"       # standard | deep
run_bpa = True
bpa_extended = False
run_vertipaq = True
vpa_read_stats_from_data = False
run_refresh_history = True
refresh_history_top_n = 20
run_unused_objects = False
unused_objects_method = "WorkspaceMonitoring"
workspace_monitoring_days = 14
run_direct_lake_checks = True
capture_item_access_snapshot = True

# Finding thresholds
min_column_size_mb = 50.0
min_column_model_pct = 10.0
high_column_model_pct = 25.0
high_cardinality_threshold = 1_000_000
max_storage_findings_per_model = 30

# Output, authorization, and safeguards
output_schema = "smopt"             # Use "" for a non-schema Lakehouse
enforce_spn_workspace_access_precheck = True
required_workspace_roles = ("Contributor", "Member", "Admin")
default_authorized_viewer_upns = () # RLS grants are normally maintained separately
model_access_sync_mode = "none"
max_models_per_run = 50
max_retries = 2
retry_base_seconds = 5
include_default_semantic_models = False
excluded_model_names = ("ModelBPA", "Fabric Capacity Metrics")
fail_pipeline_if_any_model_fails = False
fail_pipeline_if_permission_precheck_fails = True
exit_notebook_with_summary = False

# Automatically generated per run; kept out of the member-facing parameter list.
scan_request_id = ""
requested_by_upn = (requested_by_upn_optional or "").strip()


In [ ]:
import hashlib
import json
import math
import re
import time
import traceback
import uuid
from contextlib import nullcontext
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version as package_version

import pandas as pd
import sempy
import sempy.fabric as fabric
import sempy.fabric.admin as admin
import sempy_labs as labs
from delta.tables import DeltaTable
from pyspark.sql import types as T
from sempy.fabric import set_service_principal
from sempy_labs import directlake
from sempy_labs import semantic_model as labs_semantic_model


def installed_version(distribution_name, module):
    try:
        return package_version(distribution_name)
    except PackageNotFoundError:
        return getattr(module, "__version__", "UNKNOWN")


SCANNER_VERSION = "2.1.1"
SOLUTION_STAGE = "OPPORTUNITY_DISCOVERY"
SEMANTIC_LINK_VERSION = installed_version("semantic-link-sempy", sempy)
SEMANTIC_LINK_LABS_VERSION = installed_version("semantic-link-labs", labs)
print(f"Scanner {SCANNER_VERSION} ready | semantic-link={SEMANTIC_LINK_VERSION} | semantic-link-labs={SEMANTIC_LINK_LABS_VERSION}")


In [ ]:
# ---------- Generic helpers ----------

UTC = timezone.utc
RUN_STARTED_AT = datetime.now(UTC)
SCAN_ID = str(uuid.uuid4())
REQUEST_ID = scan_request_id.strip() or SCAN_ID


def utcnow():
    return datetime.now(UTC)


def stable_id(*parts):
    payload = "|".join("" if p is None else str(p) for p in parts)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def clean_string(value, max_len=8000):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    return str(value)[:max_len]


def json_safe(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (datetime, pd.Timestamp)):
        return value.isoformat()
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    return str(value)


def json_dumps(value):
    return json.dumps(value, default=json_safe, ensure_ascii=False, sort_keys=True)


def canon(text):
    return re.sub(r"[^a-z0-9]+", "_", str(text).strip().lower()).strip("_")


def canonical_record(row):
    if hasattr(row, "to_dict"):
        row = row.to_dict()
    return {canon(k): v for k, v in dict(row).items()}


def pick(record, aliases, default=None):
    for alias in aliases:
        key = canon(alias)
        if key in record:
            value = record[key]
            if value is not None and not (isinstance(value, float) and math.isnan(value)):
                return value
    return default


def as_bool(value, default=False):
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


_SIZE_UNITS = {"b": 1, "kb": 1024, "mb": 1024**2, "gb": 1024**3, "tb": 1024**4}


def as_number(value, default=None):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return default
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).replace(",", "").strip()
    match = re.fullmatch(r"(-?\d+(?:\.\d+)?)\s*([kmgt]?b)?", text, flags=re.I)
    if not match:
        return default
    number = float(match.group(1))
    unit = (match.group(2) or "").lower()
    return number * _SIZE_UNITS.get(unit, 1)


def as_int(value, default=None):
    number = as_number(value, default=None)
    return default if number is None else int(number)


def as_timestamp(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    ts = pd.to_datetime(value, utc=True, errors="coerce")
    return None if pd.isna(ts) else ts.to_pydatetime()


def duration_seconds(start_value, end_value):
    start = as_timestamp(start_value)
    end = as_timestamp(end_value)
    return None if start is None or end is None else max(0.0, (end - start).total_seconds())


def truncate_error(exc):
    return clean_string(f"{type(exc).__name__}: {exc}", max_len=4000)


def error_category(exc):
    text = str(exc).lower()
    if any(x in text for x in ["401", "unauthorized", "authentication"]):
        return "AUTHENTICATION"
    if any(x in text for x in ["403", "forbidden", "readwrite", "permission", "not authorized"]):
        return "AUTHORIZATION"
    if any(x in text for x in ["429", "throttl", "too many requests"]):
        return "THROTTLING"
    if any(x in text for x in ["timeout", "timed out"]):
        return "TIMEOUT"
    if any(x in text for x in ["not found", "doesn't exist", "does not exist"]):
        return "NOT_FOUND"
    return "ANALYSIS_ERROR"


def is_retryable(exc):
    category = error_category(exc)
    text = str(exc).lower()
    return category in {"THROTTLING", "TIMEOUT"} or any(x in text for x in ["500", "502", "503", "504"])


def with_retry(label, function):
    for attempt in range(max_retries + 1):
        try:
            return function()
        except Exception as exc:
            if attempt >= max_retries or not is_retryable(exc):
                raise
            wait_seconds = retry_base_seconds * (2**attempt)
            print(f"{label}: retry {attempt + 1}/{max_retries} in {wait_seconds}s ({error_category(exc)})")
            time.sleep(wait_seconds)


def validate_uuid(value, field_name):
    try:
        return str(uuid.UUID(str(value)))
    except Exception as exc:
        raise ValueError(f"{field_name} must be a GUID: {value}") from exc


In [ ]:
# ---------- Authentication, simple scope parsing, permission precheck, target resolution ----------

def authentication_context():
    mode = auth_mode.strip().lower()
    if mode == "user":
        return nullcontext()

    if mode == "spn_secret":
        if not allow_plaintext_spn_secret:
            raise ValueError("spn_secret is test-only. Set allow_plaintext_spn_secret=True explicitly.")
        if not all([spn_tenant_id, spn_client_id, spn_client_secret]):
            raise ValueError("spn_secret requires tenant ID, client ID and client secret.")
        return set_service_principal(
            tenant_id=spn_tenant_id,
            client_id=spn_client_id,
            client_secret=spn_client_secret,
        )

    if mode == "spn_keyvault":
        if not key_vault_name_or_uri or not kv_client_secret_name:
            raise ValueError("spn_keyvault requires key_vault_name_or_uri and kv_client_secret_name.")
        tenant_arg = (
            (key_vault_name_or_uri, kv_tenant_id_secret_name)
            if kv_tenant_id_secret_name
            else spn_tenant_id
        )
        client_arg = (
            (key_vault_name_or_uri, kv_client_id_secret_name)
            if kv_client_id_secret_name
            else spn_client_id
        )
        if not tenant_arg or not client_arg:
            raise ValueError("Provide tenant/client IDs directly or as Key Vault secret references.")
        return set_service_principal(
            tenant_id=tenant_arg,
            client_id=client_arg,
            client_secret=(key_vault_name_or_uri, kv_client_secret_name),
        )

    raise ValueError("auth_mode must be user, spn_secret or spn_keyvault.")


def parse_guid_list(value, field_name, required=False):
    """Accept comma, semicolon, whitespace, or newline separated GUIDs."""
    if value is None:
        raw_values = []
    elif isinstance(value, (list, tuple, set)):
        raw_values = [str(item).strip() for item in value]
    else:
        raw_values = re.split(r"[,;\s]+", str(value).strip())

    values = []
    seen = set()
    for raw_value in raw_values:
        if not raw_value:
            continue
        normalized = validate_uuid(raw_value, field_name).lower()
        if normalized not in seen:
            seen.add(normalized)
            values.append(normalized)
    if required and not values:
        raise ValueError(f"{field_name} is required. Enter at least one Workspace ID.")
    return values


def parse_scope():
    workspace_targets = parse_guid_list(workspace_ids, "workspace_ids", required=True)
    model_targets = parse_guid_list(model_ids_optional, "model_ids_optional", required=False)
    return workspace_targets, model_targets


def normalize_upns(values):
    if values is None:
        return []
    if isinstance(values, str):
        values = re.split(r"[,;\s]+", values.strip())
    return sorted({str(v).strip().lower() for v in values if str(v).strip()})


valid_workspace_roles = {"ADMIN", "MEMBER", "CONTRIBUTOR", "VIEWER"}
REQUIRED_WORKSPACE_ROLES = {
    str(role).strip().upper() for role in required_workspace_roles if str(role).strip()
}
if not REQUIRED_WORKSPACE_ROLES or not REQUIRED_WORKSPACE_ROLES.issubset(valid_workspace_roles):
    raise ValueError(f"required_workspace_roles must use values from {sorted(valid_workspace_roles)}.")


def item_columns(df):
    mapping = {canon(c): c for c in df.columns}
    id_col = next((mapping[k] for k in ["id", "item_id", "dataset_id", "semantic_model_id"] if k in mapping), None)
    name_col = next((mapping[k] for k in ["display_name", "name", "item_name", "dataset_name", "semantic_model_name"] if k in mapping), None)
    if not id_col or not name_col:
        raise ValueError(f"Unable to locate ID/name columns in list_items result: {list(df.columns)}")
    return id_col, name_col


def guid_values(value):
    text = json_dumps(value)
    matches = re.findall(
        r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}",
        text,
    )
    return {str(uuid.UUID(match)).lower() for match in matches}


WORKSPACE_PRECHECK_CACHE = {}


def precheck_workspace_access(workspace_id):
    workspace_id = validate_uuid(workspace_id, "workspace_id")
    cache_key = workspace_id.lower()
    if cache_key in WORKSPACE_PRECHECK_CACHE:
        return WORKSPACE_PRECHECK_CACHE[cache_key]

    mode = auth_mode.strip().lower()
    result = {
        "status": "NOT_CHECKED_USER_MODE" if mode == "user" else "NOT_ENFORCED",
        "workspace_name": None,
        "workspace_role": None,
        "message": "User-mode scans rely on runtime authorization." if mode == "user" else "SPN precheck is disabled.",
    }
    if mode == "user" or not enforce_spn_workspace_access_precheck:
        WORKSPACE_PRECHECK_CACHE[cache_key] = result
        return result

    expected_ids = guid_values([spn_object_id, spn_client_id])
    if not expected_ids:
        result.update({
            "status": "FAILED_SCANNER_ID_REQUIRED",
            "message": "Provide spn_object_id, or provide spn_client_id directly instead of only a Key Vault reference.",
        })
        WORKSPACE_PRECHECK_CACHE[cache_key] = result
        return result

    try:
        access_df = with_retry(
            "workspace_access_precheck",
            lambda: admin.list_workspace_access_details(workspace=workspace_id),
        )
        matched = None
        for _, raw_row in access_df.iterrows():
            record = canonical_record(raw_row)
            if result["workspace_name"] is None:
                result["workspace_name"] = clean_string(pick(record, ["Workspace Name", "Name"]), 1000)
            principal_type = str(pick(record, ["User Type", "Principal Type"], "")).replace(" ", "").lower()
            row_ids = guid_values({
                "principal_id": pick(record, ["User Id", "Principal Id", "Graph Id", "Identifier"]),
                "principal_details": pick(record, ["User Details", "Service Principal Details"]),
            })
            if principal_type in {"serviceprincipal", "app", "serviceprincipalprofile"} and row_ids.intersection(expected_ids):
                matched = record
                break

        if matched is None:
            result.update({
                "status": "FAILED_NOT_IN_WORKSPACE",
                "message": "The scanner SPN was not found as a direct principal in the target workspace.",
            })
        else:
            role = clean_string(pick(matched, ["Workspace Role", "Role", "Group User Access Right"]), 200)
            normalized_role = (role or "").strip().upper()
            if normalized_role in REQUIRED_WORKSPACE_ROLES:
                result.update({
                    "status": "PASSED",
                    "workspace_role": role,
                    "message": f"Scanner SPN has accepted workspace role: {role}.",
                })
            else:
                result.update({
                    "status": "FAILED_INSUFFICIENT_ROLE",
                    "workspace_role": role,
                    "message": f"Scanner SPN role '{role or 'UNKNOWN'}' is not in {sorted(REQUIRED_WORKSPACE_ROLES)}.",
                })
    except Exception as exc:
        result.update({
            "status": "ERROR",
            "message": f"Workspace access precheck failed: {truncate_error(exc)}",
        })

    WORKSPACE_PRECHECK_CACHE[cache_key] = result
    return result


def workspace_name_from_admin(workspace_id, precheck):
    if precheck.get("workspace_name"):
        return precheck["workspace_name"]
    workspaces_df = admin.list_workspaces(workspace=workspace_id)
    if workspaces_df is None or workspaces_df.empty:
        return workspace_id
    record = canonical_record(workspaces_df.iloc[0])
    return clean_string(pick(record, ["Name", "Workspace Name"], workspace_id), 1000)


def resolve_targets():
    workspace_targets, requested_model_ids = parse_scope()
    excluded_names = {str(name).strip().lower() for name in excluded_model_names}
    default_viewers = normalize_upns(default_authorized_viewer_upns)
    inventory = {}

    def load_workspace(ws_id):
        ws_id = validate_uuid(ws_id, "workspace_id")
        precheck = precheck_workspace_access(ws_id)
        if auth_mode.strip().lower() == "user":
            ws_name, normalized_ws_id = fabric.resolve_workspace_name_and_id(ws_id)
            items = fabric.list_items(item_type="SemanticModel", workspace=normalized_ws_id)
        else:
            normalized_ws_id = ws_id
            ws_name = workspace_name_from_admin(ws_id, precheck)
            items = admin.list_items(item_type="SemanticModel", workspace=normalized_ws_id)
        id_col, name_col = item_columns(items)
        model_map = {
            str(row[id_col]).lower(): str(row[name_col])
            for _, row in items.iterrows()
        }
        return str(normalized_ws_id), str(ws_name), model_map, precheck

    for workspace_id in workspace_targets:
        normalized_ws_id, ws_name, model_map, precheck = load_workspace(workspace_id)
        for model_id, model_name in model_map.items():
            inventory.setdefault(model_id, []).append({
                "workspace_id": normalized_ws_id,
                "workspace_name": ws_name,
                "model_id": model_id,
                "model_name": model_name,
                "authorized_viewer_upns": default_viewers,
                "scope_source": "MODEL" if requested_model_ids else "WORKSPACE",
                "permission_precheck_status": precheck["status"],
                "scanner_workspace_role": precheck["workspace_role"],
                "permission_precheck_message": precheck["message"],
            })

    if requested_model_ids:
        targets = []
        missing = []
        ambiguous = []
        for model_id in requested_model_ids:
            matches = inventory.get(model_id, [])
            if not matches:
                missing.append(model_id)
            elif len(matches) > 1:
                ambiguous.append(model_id)
            else:
                target = matches[0]
                if target["model_name"].strip().lower() in excluded_names:
                    raise ValueError(f"Model {model_id} is excluded by platform configuration.")
                targets.append(target)
        if missing:
            raise ValueError(
                "The following Model IDs were not found in the supplied workspaces: " + ", ".join(missing)
            )
        if ambiguous:
            raise ValueError("A Model ID matched more than one supplied workspace: " + ", ".join(ambiguous))
    else:
        targets = [
            matches[0]
            for matches in inventory.values()
            if matches and matches[0]["model_name"].strip().lower() not in excluded_names
        ]

    if not include_default_semantic_models:
        eligible = []
        for target in targets:
            can_open_model = target["permission_precheck_status"] in {
                "PASSED", "NOT_CHECKED_USER_MODE", "NOT_ENFORCED"
            }
            if not can_open_model:
                eligible.append(target)
                continue
            is_default = labs.is_default_semantic_model(
                dataset=target["model_name"],
                workspace=target["workspace_id"],
            )
            if not is_default:
                eligible.append(target)
        targets = eligible

    if len(targets) > max_models_per_run:
        raise ValueError(
            f"The scope contains {len(targets)} models, above the configured limit of {max_models_per_run}. "
            "Enter selected Model IDs or ask the platform owner to raise the safeguard."
        )
    if not targets:
        raise ValueError("No eligible semantic models were found in the supplied scope.")
    return targets


if not requested_by_upn.strip():
    print("Note: requester email is blank; the scan can run, but the audit field will be empty.")
if spn_object_id:
    validate_uuid(spn_object_id, "spn_object_id")
if analysis_profile.strip().lower() == "deep":
    run_unused_objects = True
if not run_bpa and not run_vertipaq:
    raise ValueError("At least one core analysis (BPA or VertiPaq) must be enabled.")
if model_access_sync_mode not in {"none", "merge", "replace_scanner_managed"}:
    raise ValueError("Invalid model_access_sync_mode.")


In [ ]:
# ---------- Delta table contracts and idempotent writers ----------

RUN_SCHEMA = T.StructType([
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("request_id", T.StringType(), False),
    T.StructField("requested_by_upn", T.StringType()),
    T.StructField("auth_mode", T.StringType()),
    T.StructField("analysis_profile", T.StringType()),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("solution_stage", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("started_at", T.TimestampType()),
    T.StructField("completed_at", T.TimestampType()),
    T.StructField("target_count", T.IntegerType()),
    T.StructField("success_count", T.IntegerType()),
    T.StructField("partial_count", T.IntegerType()),
    T.StructField("failed_count", T.IntegerType()),
    T.StructField("skipped_count", T.IntegerType()),
    T.StructField("semantic_link_version", T.StringType()),
    T.StructField("semantic_link_labs_version", T.StringType()),
    T.StructField("parameters_hash", T.StringType()),
    T.StructField("error_category", T.StringType()),
    T.StructField("error_message", T.StringType()),
])

MODEL_SCAN_SCHEMA = T.StructType([
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("model_id", T.StringType(), False),
    T.StructField("model_name", T.StringType()),
    T.StructField("scope_source", T.StringType()),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("solution_stage", T.StringType()),
    T.StructField("permission_precheck_status", T.StringType()),
    T.StructField("scanner_workspace_role", T.StringType()),
    T.StructField("permission_precheck_message", T.StringType()),
    T.StructField("capacity_id", T.StringType()),
    T.StructField("capacity_name", T.StringType()),
    T.StructField("storage_mode", T.StringType()),
    T.StructField("model_size_bytes", T.LongType()),
    T.StructField("overall_status", T.StringType()),
    T.StructField("bpa_status", T.StringType()),
    T.StructField("vpa_status", T.StringType()),
    T.StructField("refresh_status", T.StringType()),
    T.StructField("usage_status", T.StringType()),
    T.StructField("direct_lake_status", T.StringType()),
    T.StructField("access_snapshot_status", T.StringType()),
    T.StructField("finding_count", T.IntegerType()),
    T.StructField("started_at", T.TimestampType()),
    T.StructField("completed_at", T.TimestampType()),
    T.StructField("duration_seconds", T.DoubleType()),
    T.StructField("error_json", T.StringType()),
])

FINDING_SCHEMA = T.StructType([
    T.StructField("finding_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("solution_stage", T.StringType()),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("model_name", T.StringType()),
    T.StructField("source", T.StringType()),
    T.StructField("category", T.StringType()),
    T.StructField("rule_id", T.StringType()),
    T.StructField("rule_name", T.StringType()),
    T.StructField("severity", T.StringType()),
    T.StructField("confidence", T.StringType()),
    T.StructField("impact_area", T.StringType()),
    T.StructField("object_type", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("object_name", T.StringType()),
    T.StructField("finding_text", T.StringType()),
    T.StructField("recommended_action", T.StringType()),
    T.StructField("technical_evidence", T.StringType()),
    T.StructField("evidence_json", T.StringType()),
    T.StructField("estimated_saving_bytes_low", T.LongType()),
    T.StructField("estimated_saving_bytes_high", T.LongType()),
    T.StructField("reclaimable_upper_bound_bytes", T.LongType()),
    T.StructField("cu_saving_status", T.StringType()),
    T.StructField("benefit_validation_status", T.StringType()),
    T.StructField("change_risk", T.StringType()),
    T.StructField("validation_required", T.BooleanType()),
    T.StructField("documentation_url", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

VPA_COLUMN_SCHEMA = T.StructType([
    T.StructField("evidence_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("column_name", T.StringType()),
    T.StructField("data_type", T.StringType()),
    T.StructField("encoding", T.StringType()),
    T.StructField("cardinality", T.LongType()),
    T.StructField("data_size_bytes", T.LongType()),
    T.StructField("dictionary_size_bytes", T.LongType()),
    T.StructField("hierarchy_size_bytes", T.LongType()),
    T.StructField("total_size_bytes", T.LongType()),
    T.StructField("model_size_pct", T.DoubleType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

VPA_TABLE_SCHEMA = T.StructType([
    T.StructField("evidence_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("row_count", T.LongType()),
    T.StructField("data_size_bytes", T.LongType()),
    T.StructField("dictionary_size_bytes", T.LongType()),
    T.StructField("hierarchy_size_bytes", T.LongType()),
    T.StructField("total_size_bytes", T.LongType()),
    T.StructField("model_size_pct", T.DoubleType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

OBJECT_USAGE_SCHEMA = T.StructType([
    T.StructField("usage_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("object_type", T.StringType()),
    T.StructField("object_name", T.StringType()),
    T.StructField("is_used", T.BooleanType()),
    T.StructField("usage_count", T.LongType()),
    T.StructField("usage_method", T.StringType()),
    T.StructField("usage_window_days", T.IntegerType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

REFRESH_SCHEMA = T.StructType([
    T.StructField("refresh_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("request_id", T.StringType()),
    T.StructField("refresh_type", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("start_time", T.TimestampType()),
    T.StructField("end_time", T.TimestampType()),
    T.StructField("duration_seconds", T.DoubleType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("captured_at", T.TimestampType()),
])

DIRECT_LAKE_SCHEMA = T.StructType([
    T.StructField("check_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("check_type", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("reason", T.StringType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

ACCESS_SNAPSHOT_SCHEMA = T.StructType([
    T.StructField("access_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("principal_id", T.StringType()),
    T.StructField("principal_name", T.StringType()),
    T.StructField("principal_type", T.StringType()),
    T.StructField("principal_upn", T.StringType()),
    T.StructField("permissions", T.StringType()),
    T.StructField("additional_permissions", T.StringType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("captured_at", T.TimestampType()),
])

MODEL_ACCESS_SCHEMA = T.StructType([
    T.StructField("access_key", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("principal_upn", T.StringType()),
    T.StructField("source", T.StringType()),
    T.StructField("is_active", T.BooleanType()),
    T.StructField("valid_from", T.TimestampType()),
    T.StructField("valid_to", T.TimestampType()),
    T.StructField("updated_by", T.StringType()),
    T.StructField("updated_at", T.TimestampType()),
])

DIM_MODEL_SCHEMA = T.StructType([
    T.StructField("model_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("model_name", T.StringType()),
    T.StructField("capacity_id", T.StringType()),
    T.StructField("capacity_name", T.StringType()),
    T.StructField("storage_mode", T.StringType()),
    T.StructField("latest_scan_id", T.StringType()),
    T.StructField("last_scan_status", T.StringType()),
    T.StructField("last_scanned_at", T.TimestampType()),
])

TABLES = {
    "scan_run": (RUN_SCHEMA, ["scan_id"]),
    "model_scan": (MODEL_SCAN_SCHEMA, ["scan_id", "model_id"]),
    "finding": (FINDING_SCHEMA, ["finding_id"]),
    "vpa_column": (VPA_COLUMN_SCHEMA, ["evidence_id"]),
    "vpa_table": (VPA_TABLE_SCHEMA, ["evidence_id"]),
    "object_usage": (OBJECT_USAGE_SCHEMA, ["usage_id"]),
    "refresh": (REFRESH_SCHEMA, ["refresh_id"]),
    "direct_lake": (DIRECT_LAKE_SCHEMA, ["check_id"]),
    "item_access_snapshot": (ACCESS_SNAPSHOT_SCHEMA, ["access_id"]),
    "model_access": (MODEL_ACCESS_SCHEMA, ["access_key"]),
    "dim_model": (DIM_MODEL_SCHEMA, ["model_id"]),
}


def table_name(logical_name):
    physical = f"smopt_{logical_name}"
    return f"{output_schema}.{physical}" if output_schema else physical


def ensure_tables():
    if output_schema:
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", output_schema):
            raise ValueError("output_schema contains invalid characters.")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{output_schema}`")
    for logical_name, (schema, _) in TABLES.items():
        name = table_name(logical_name)
        if not spark.catalog.tableExists(name):
            spark.createDataFrame([], schema).write.format("delta").mode("errorifexists").saveAsTable(name)
            continue
        existing_columns = {field.name.lower() for field in spark.table(name).schema.fields}
        missing_fields = [field for field in schema.fields if field.name.lower() not in existing_columns]
        if missing_fields:
            additions = ", ".join(
                f"`{field.name}` {field.dataType.simpleString().upper()}"
                for field in missing_fields
            )
            spark.sql(f"ALTER TABLE {name} ADD COLUMNS ({additions})")


def upsert_rows(logical_name, rows):
    if not rows:
        return
    schema, keys = TABLES[logical_name]
    source = spark.createDataFrame(rows, schema=schema)
    name = table_name(logical_name)
    condition = " AND ".join(f"t.`{key}` = s.`{key}`" for key in keys)
    (
        DeltaTable.forName(spark, name)
        .alias("t")
        .merge(source.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


def safe_parameters_hash():
    safe = {
        "workspace_ids": workspace_ids,
        "model_ids_optional": model_ids_optional,
        "analysis_profile": analysis_profile,
        "auth_mode": auth_mode,
        "scanner_version": SCANNER_VERSION,
        "solution_stage": SOLUTION_STAGE,
        "spn_object_id": spn_object_id,
        "enforce_spn_workspace_access_precheck": enforce_spn_workspace_access_precheck,
        "required_workspace_roles": sorted(REQUIRED_WORKSPACE_ROLES),
        "run_bpa": run_bpa,
        "bpa_extended": bpa_extended,
        "run_vertipaq": run_vertipaq,
        "vpa_read_stats_from_data": vpa_read_stats_from_data,
        "run_refresh_history": run_refresh_history,
        "run_unused_objects": run_unused_objects,
        "unused_objects_method": unused_objects_method,
        "run_direct_lake_checks": run_direct_lake_checks,
        "capture_item_access_snapshot": capture_item_access_snapshot,
        "thresholds": [min_column_size_mb, min_column_model_pct, high_column_model_pct, high_cardinality_threshold],
    }
    return stable_id(json_dumps(safe))


In [ ]:
# ---------- Analysis result normalizers ----------

def finding_base(target, source, rule_name, object_type=None, table_name_value=None, object_name=None):
    now = utcnow()
    rule_id = stable_id(source, rule_name)
    return {
        "finding_id": stable_id(SCAN_ID, target["model_id"], source, rule_name, table_name_value, object_name),
        "scan_id": SCAN_ID,
        "scanner_version": SCANNER_VERSION,
        "solution_stage": SOLUTION_STAGE,
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_id": target["model_id"],
        "model_name": target["model_name"],
        "source": source,
        "category": None,
        "rule_id": rule_id,
        "rule_name": clean_string(rule_name, 1000),
        "severity": "INFO",
        "confidence": "MEDIUM",
        "impact_area": "MODEL_QUALITY",
        "object_type": clean_string(object_type, 200),
        "table_name": clean_string(table_name_value, 1000),
        "object_name": clean_string(object_name, 1000),
        "finding_text": None,
        "recommended_action": None,
        "technical_evidence": None,
        "evidence_json": None,
        "estimated_saving_bytes_low": None,
        "estimated_saving_bytes_high": None,
        "reclaimable_upper_bound_bytes": None,
        "cu_saving_status": "NOT_ESTIMATED_STAGE_2_REQUIRED",
        "benefit_validation_status": "NOT_STARTED",
        "change_risk": "MEDIUM",
        "validation_required": True,
        "documentation_url": None,
        "detected_at": now,
    }


def normalize_bpa(target, bpa_df):
    findings = []
    if bpa_df is None or bpa_df.empty:
        return findings
    for _, raw_row in bpa_df.iterrows():
        record = canonical_record(raw_row)
        rule_name = clean_string(pick(record, ["Rule Name", "Rule", "Name"], "BPA rule"), 1000)
        object_type = clean_string(pick(record, ["Scope", "Object Type", "ObjectType"]), 200)
        table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
        object_value = clean_string(pick(record, ["Object Name", "Object", "ObjectName"]), 1000)
        description = clean_string(pick(record, ["Description", "Finding", "Message"]), 8000)
        severity = str(pick(record, ["Severity"], "Warning")).strip().upper()
        result = finding_base(target, "BPA", rule_name, object_type, table_value, object_value)
        result.update({
            "category": clean_string(pick(record, ["Category"]), 500),
            "severity": severity if severity in {"INFO", "WARNING", "ERROR"} else "WARNING",
            "confidence": "HIGH",
            "impact_area": "PERFORMANCE" if str(pick(record, ["Category"], "")).lower() == "performance" else "MODEL_QUALITY",
            "finding_text": description or rule_name,
            "recommended_action": description,
            "technical_evidence": f"Deterministic BPA rule violation: {rule_name}",
            "evidence_json": json_dumps(raw_row.to_dict()),
            "change_risk": "MEDIUM",
            "documentation_url": clean_string(pick(record, ["URL", "Documentation URL"]), 2000),
        })
        findings.append(result)
    return findings


def normalize_vpa(target, vpa_dict, model_size_bytes):
    column_rows, table_rows, storage_findings = [], [], []
    now = utcnow()
    model_size = model_size_bytes or 0

    columns_df = next((df for key, df in vpa_dict.items() if canon(key) == "columns"), pd.DataFrame())
    for _, raw_row in columns_df.iterrows():
        record = canonical_record(raw_row)
        table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
        column_value = clean_string(pick(record, ["Column Name", "Column"]), 1000)
        total_size = as_int(pick(record, ["Total Size", "Total Size Bytes"]))
        data_size = as_int(pick(record, ["Data Size", "Data Size Bytes"]))
        dictionary_size = as_int(pick(record, ["Dictionary Size", "Dictionary Size Bytes"]))
        hierarchy_size = as_int(pick(record, ["Hierarchy Size", "Hierarchy Size Bytes"]))
        cardinality = as_int(pick(record, ["Cardinality", "Column Cardinality"]))
        pct = (100.0 * total_size / model_size) if total_size is not None and model_size > 0 else None
        evidence_id = stable_id(SCAN_ID, target["model_id"], "VPA_COLUMN", table_value, column_value)
        column_rows.append({
            "evidence_id": evidence_id,
            "scan_id": SCAN_ID,
            "workspace_id": target["workspace_id"],
            "model_id": target["model_id"],
            "table_name": table_value,
            "column_name": column_value,
            "data_type": clean_string(pick(record, ["Data Type", "Type"]), 200),
            "encoding": clean_string(pick(record, ["Encoding", "Encoding Hint"]), 200),
            "cardinality": cardinality,
            "data_size_bytes": data_size,
            "dictionary_size_bytes": dictionary_size,
            "hierarchy_size_bytes": hierarchy_size,
            "total_size_bytes": total_size,
            "model_size_pct": pct,
            "raw_json": json_dumps(raw_row.to_dict()),
            "detected_at": now,
        })

    tables_df = next((df for key, df in vpa_dict.items() if canon(key) in {"tables", "table"}), pd.DataFrame())
    for _, raw_row in tables_df.iterrows():
        record = canonical_record(raw_row)
        table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
        total_size = as_int(pick(record, ["Total Size", "Total Size Bytes"]))
        pct = (100.0 * total_size / model_size) if total_size is not None and model_size > 0 else None
        table_rows.append({
            "evidence_id": stable_id(SCAN_ID, target["model_id"], "VPA_TABLE", table_value),
            "scan_id": SCAN_ID,
            "workspace_id": target["workspace_id"],
            "model_id": target["model_id"],
            "table_name": table_value,
            "row_count": as_int(pick(record, ["Row Count", "Rows"])),
            "data_size_bytes": as_int(pick(record, ["Data Size", "Data Size Bytes"])),
            "dictionary_size_bytes": as_int(pick(record, ["Dictionary Size", "Dictionary Size Bytes"])),
            "hierarchy_size_bytes": as_int(pick(record, ["Hierarchy Size", "Hierarchy Size Bytes"])),
            "total_size_bytes": total_size,
            "model_size_pct": pct,
            "raw_json": json_dumps(raw_row.to_dict()),
            "detected_at": now,
        })

    candidates = []
    min_bytes = int(min_column_size_mb * 1024 * 1024)
    for row in column_rows:
        size = row["total_size_bytes"] or 0
        pct = row["model_size_pct"] or 0.0
        cardinality = row["cardinality"] or 0
        if (size >= min_bytes and pct >= min_column_model_pct) or (size >= min_bytes and cardinality >= high_cardinality_threshold):
            candidates.append(row)
    candidates.sort(key=lambda x: x["total_size_bytes"] or 0, reverse=True)

    for row in candidates[:max_storage_findings_per_model]:
        data_type = (row["data_type"] or "").lower()
        rule_name = "Large high-cardinality column" if (row["cardinality"] or 0) >= high_cardinality_threshold else "Large storage contributor"
        finding = finding_base(target, "VPA_HEURISTIC", rule_name, "Column", row["table_name"], row["column_name"])
        size_mb = (row["total_size_bytes"] or 0) / 1024 / 1024
        pct = row["model_size_pct"]
        evidence_text = f"Column size={size_mb:.2f} MB; model share={pct:.2f}%" if pct is not None else f"Column size={size_mb:.2f} MB"
        if row["cardinality"] is not None:
            evidence_text += f"; cardinality={row['cardinality']:,}"
        recommendations = ["Confirm the column is used by reports, measures, relationships, external XMLA clients, and exports before changing it."]
        if "date" in data_type and "time" in data_type:
            recommendations.append("If business precision permits, reduce DateTime precision or split Date and Time usage to reduce cardinality.")
        elif any(x in data_type for x in ["string", "text"]):
            recommendations.append("Review long/high-cardinality text in fact tables; consider removal, normalization, or a surrogate key where semantically valid.")
        else:
            recommendations.append("Review granularity, data type, unused values, and whether aggregation can satisfy the reporting requirement.")
        finding.update({
            "category": "Storage",
            "severity": "WARNING" if (pct or 0) < high_column_model_pct else "ERROR",
            "confidence": "MEDIUM",
            "impact_area": "MODEL_SIZE",
            "finding_text": "The column is a material contributor to model memory. This is a prioritization signal, not proof that the column is unnecessary.",
            "recommended_action": " ".join(recommendations),
            "technical_evidence": evidence_text,
            "evidence_json": row["raw_json"],
            "reclaimable_upper_bound_bytes": row["total_size_bytes"],
            "change_risk": "HIGH",
        })
        storage_findings.append(finding)

    return column_rows, table_rows, storage_findings


def detect_storage_mode(vpa_dict):
    partitions_df = next((df for key, df in vpa_dict.items() if canon(key) in {"partitions", "partition"}), pd.DataFrame())
    if partitions_df.empty:
        return "UNKNOWN"
    values = set()
    for _, raw_row in partitions_df.iterrows():
        record = canonical_record(raw_row)
        mode = pick(record, ["Mode", "Storage Mode", "Partition Mode"])
        if mode is not None:
            values.add(str(mode).strip())
    return ", ".join(sorted(values)) if values else "UNKNOWN"


def normalize_usage(target, usage_df, vpa_columns):
    rows, findings = [], []
    if usage_df is None or usage_df.empty:
        return rows, findings
    size_lookup = {
        ((row["table_name"] or "").lower(), (row["column_name"] or "").lower()): row["total_size_bytes"]
        for row in vpa_columns
    }
    for _, raw_row in usage_df.iterrows():
        record = canonical_record(raw_row)
        table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
        object_type = clean_string(pick(record, ["Object Type", "Type"]), 200)
        object_value = clean_string(pick(record, ["Object Name", "Object"]), 1000)
        is_used = as_bool(pick(record, ["IsUsed", "Is Used"]), default=True)
        usage_count = as_int(pick(record, ["UsageCount", "Usage Count"]), default=0)
        usage_row = {
            "usage_id": stable_id(SCAN_ID, target["model_id"], "USAGE", table_value, object_type, object_value),
            "scan_id": SCAN_ID,
            "workspace_id": target["workspace_id"],
            "model_id": target["model_id"],
            "table_name": table_value,
            "object_type": object_type,
            "object_name": object_value,
            "is_used": is_used,
            "usage_count": usage_count,
            "usage_method": unused_objects_method,
            "usage_window_days": workspace_monitoring_days if unused_objects_method == "WorkspaceMonitoring" else None,
            "raw_json": json_dumps(raw_row.to_dict()),
            "detected_at": utcnow(),
        }
        rows.append(usage_row)
        if not is_used:
            upper_bound = size_lookup.get(((table_value or "").lower(), (object_value or "").lower()))
            finding = finding_base(target, "UNUSED_OBJECT_ANALYSIS", "Unused semantic model object", object_type, table_value, object_value)
            evidence_scope = (
                f"No references in {workspace_monitoring_days} days of Workspace Monitoring queries."
                if unused_objects_method == "WorkspaceMonitoring"
                else "No references found in downstream reports available in PBIR format."
            )
            finding.update({
                "category": "Usage",
                "severity": "WARNING" if upper_bound and upper_bound >= min_column_size_mb * 1024 * 1024 else "INFO",
                "confidence": "MEDIUM",
                "impact_area": "MODEL_SIZE" if object_type and object_type.lower() in {"column", "table"} else "MAINTAINABILITY",
                "finding_text": "The object was not observed in the selected usage evidence scope.",
                "recommended_action": "Validate external tools, thin reports in other workspaces, Analyze in Excel, paginated reports, subscriptions, APIs, and future-use requirements before removal.",
                "technical_evidence": evidence_scope,
                "evidence_json": usage_row["raw_json"],
                "reclaimable_upper_bound_bytes": upper_bound,
                "change_risk": "HIGH",
            })
            findings.append(finding)
    return rows, findings


In [ ]:
# ---------- Per-model scanner ----------

def skipped_permission_result(target):
    started = utcnow()
    completed = utcnow()
    message = target.get("permission_precheck_message") or "SPN workspace permission precheck did not pass."
    gated_status = "SKIPPED_PERMISSION"
    model_row = {
        "scan_id": SCAN_ID,
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_id": target["model_id"],
        "model_name": target["model_name"],
        "scope_source": target["scope_source"],
        "scanner_version": SCANNER_VERSION,
        "solution_stage": SOLUTION_STAGE,
        "permission_precheck_status": target.get("permission_precheck_status"),
        "scanner_workspace_role": target.get("scanner_workspace_role"),
        "permission_precheck_message": message,
        "capacity_id": None,
        "capacity_name": None,
        "storage_mode": "UNKNOWN",
        "model_size_bytes": None,
        "overall_status": gated_status,
        "bpa_status": gated_status if run_bpa else "NOT_RUN",
        "vpa_status": gated_status if run_vertipaq else "NOT_RUN",
        "refresh_status": gated_status if run_refresh_history else "NOT_RUN",
        "usage_status": gated_status if run_unused_objects else "NOT_RUN",
        "direct_lake_status": gated_status if run_direct_lake_checks else "NOT_RUN",
        "access_snapshot_status": "NOT_RUN_PERMISSION_PRECHECK",
        "finding_count": 0,
        "started_at": started,
        "completed_at": completed,
        "duration_seconds": (completed - started).total_seconds(),
        "error_json": json_dumps({"permission_precheck": message}),
    }
    dim_model_row = {
        "model_id": target["model_id"],
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_name": target["model_name"],
        "capacity_id": None,
        "capacity_name": None,
        "storage_mode": "UNKNOWN",
        "latest_scan_id": SCAN_ID,
        "last_scan_status": gated_status,
        "last_scanned_at": completed,
    }
    return {
        "model_row": model_row,
        "dim_model_row": dim_model_row,
        "findings": [],
        "vpa_columns": [],
        "vpa_tables": [],
        "usage_rows": [],
        "refresh_rows": [],
        "direct_lake_rows": [],
        "access_rows": [],
    }


def scan_one_model(target):
    if (
        auth_mode.strip().lower() != "user"
        and enforce_spn_workspace_access_precheck
        and target.get("permission_precheck_status") != "PASSED"
    ):
        print(
            f"Skipping {target['workspace_name']} / {target['model_name']}: "
            f"{target.get('permission_precheck_status')} — {target.get('permission_precheck_message')}"
        )
        return skipped_permission_result(target)

    started = utcnow()
    statuses = {
        "bpa": "NOT_RUN",
        "vpa": "NOT_RUN",
        "refresh": "NOT_RUN",
        "usage": "NOT_RUN",
        "direct_lake": "NOT_RUN",
        "access_snapshot": "NOT_RUN",
    }
    errors = {}
    findings, vpa_columns, vpa_tables = [], [], []
    usage_rows, refresh_rows, direct_lake_rows, access_rows = [], [], [], []
    capacity_id = capacity_name = None
    storage_mode = "UNKNOWN"
    model_size_bytes = None

    print(f"Scanning {target['workspace_name']} / {target['model_name']} ({target['model_id']})")

    try:
        capacity_id = clean_string(labs.get_capacity_id(workspace=target["workspace_id"]), 200)
        capacity_name = clean_string(labs.get_capacity_name(workspace=target["workspace_id"]), 500)
    except Exception as exc:
        errors["capacity"] = truncate_error(exc)

    try:
        model_size_bytes = int(with_retry(
            "model_size",
            lambda: labs.get_semantic_model_size(dataset=target["model_name"], workspace=target["workspace_id"]),
        ))
    except Exception as exc:
        errors["model_size"] = truncate_error(exc)

    if run_bpa:
        try:
            bpa_df = with_retry(
                "bpa",
                lambda: labs.run_model_bpa(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                    return_dataframe=True,
                    export=False,
                    extended=bpa_extended,
                    check_dependencies=True,
                ),
            )
            findings.extend(normalize_bpa(target, bpa_df))
            statuses["bpa"] = "SUCCEEDED"
        except Exception as exc:
            statuses["bpa"] = "FAILED"
            errors["bpa"] = truncate_error(exc)

    vpa_dict = {}
    if run_vertipaq:
        try:
            vpa_dict = with_retry(
                "vertipaq",
                lambda: labs.vertipaq_analyzer(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                    read_stats_from_data=vpa_read_stats_from_data,
                ),
            )
            vpa_columns, vpa_tables, storage_findings = normalize_vpa(target, vpa_dict, model_size_bytes)
            findings.extend(storage_findings)
            storage_mode = detect_storage_mode(vpa_dict)
            statuses["vpa"] = "SUCCEEDED"
        except Exception as exc:
            statuses["vpa"] = "FAILED"
            errors["vpa"] = truncate_error(exc)

    if run_refresh_history:
        try:
            refresh_df = with_retry(
                "refresh_history",
                lambda: fabric.list_refresh_requests(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                    top_n=refresh_history_top_n,
                ),
            )
            for index, raw_row in refresh_df.iterrows():
                record = canonical_record(raw_row)
                upstream_request_id = clean_string(pick(record, ["Request Id", "Id", "Refresh Id"]), 500)
                start_time = as_timestamp(pick(record, ["Start Time", "StartTime", "Start Date Time"] ))
                end_time = as_timestamp(pick(record, ["End Time", "EndTime", "End Date Time"] ))
                refresh_rows.append({
                    "refresh_id": stable_id(SCAN_ID, target["model_id"], "REFRESH", upstream_request_id or index),
                    "scan_id": SCAN_ID,
                    "workspace_id": target["workspace_id"],
                    "model_id": target["model_id"],
                    "request_id": upstream_request_id,
                    "refresh_type": clean_string(pick(record, ["Refresh Type", "Type"]), 200),
                    "status": clean_string(pick(record, ["Status"]), 200),
                    "start_time": start_time,
                    "end_time": end_time,
                    "duration_seconds": duration_seconds(start_time, end_time),
                    "raw_json": json_dumps(raw_row.to_dict()),
                    "captured_at": utcnow(),
                })
            statuses["refresh"] = "SUCCEEDED"
        except Exception as exc:
            statuses["refresh"] = "FAILED"
            errors["refresh"] = truncate_error(exc)

    if run_unused_objects:
        try:
            usage_df = with_retry(
                "unused_objects",
                lambda: labs_semantic_model.find_unused_objects(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                    method=unused_objects_method,
                    workspace_monitoring_days=workspace_monitoring_days,
                    visualize=False,
                ),
            )
            usage_rows, usage_findings = normalize_usage(target, usage_df, vpa_columns)
            findings.extend(usage_findings)
            statuses["usage"] = "SUCCEEDED"
        except Exception as exc:
            statuses["usage"] = "FAILED"
            errors["usage"] = truncate_error(exc)

    is_direct_lake = "directlake" in storage_mode.replace(" ", "").lower()
    if run_direct_lake_checks and is_direct_lake:
        try:
            fallback_df = with_retry(
                "direct_lake_fallback",
                lambda: directlake.check_fallback_reason(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                ),
            )
            for index, raw_row in fallback_df.iterrows():
                record = canonical_record(raw_row)
                table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
                reason = clean_string(pick(record, ["Fallback Reason", "Reason"]), 8000)
                status_value = "ISSUE" if reason and reason.strip().lower() not in {"none", "n/a", "no fallback"} else "OK"
                direct_lake_rows.append({
                    "check_id": stable_id(SCAN_ID, target["model_id"], "DIRECT_LAKE_FALLBACK", table_value, index),
                    "scan_id": SCAN_ID,
                    "workspace_id": target["workspace_id"],
                    "model_id": target["model_id"],
                    "check_type": "FALLBACK_REASON",
                    "table_name": table_value,
                    "status": status_value,
                    "reason": reason,
                    "raw_json": json_dumps(raw_row.to_dict()),
                    "detected_at": utcnow(),
                })
                if status_value == "ISSUE":
                    finding = finding_base(target, "DIRECT_LAKE", "Direct Lake fallback risk", "Table", table_value, table_value)
                    finding.update({
                        "category": "Direct Lake",
                        "severity": "WARNING",
                        "confidence": "HIGH",
                        "impact_area": "QUERY_CU",
                        "finding_text": reason,
                        "recommended_action": "Resolve the reported fallback condition and validate query behavior and CU before/after the change.",
                        "technical_evidence": reason,
                        "evidence_json": json_dumps(raw_row.to_dict()),
                        "change_risk": "MEDIUM",
                    })
                    findings.append(finding)
            statuses["direct_lake"] = "SUCCEEDED"
        except Exception as exc:
            statuses["direct_lake"] = "FAILED"
            errors["direct_lake"] = truncate_error(exc)
    elif run_direct_lake_checks:
        statuses["direct_lake"] = "NOT_APPLICABLE"

    if capture_item_access_snapshot:
        try:
            access_df = with_retry(
                "item_access_snapshot",
                lambda: admin.list_item_access_details(
                    item=target["model_id"],
                    item_type="SemanticModel",
                    workspace=target["workspace_id"],
                ),
            )
            for index, raw_row in access_df.iterrows():
                record = canonical_record(raw_row)
                principal_id = clean_string(pick(record, ["User Id", "Graph Id", "Identifier"]), 500)
                principal_upn = clean_string(pick(record, ["User Principal Name", "Email Address"]), 1000)
                access_rows.append({
                    "access_id": stable_id(SCAN_ID, target["model_id"], principal_id, principal_upn, index),
                    "scan_id": SCAN_ID,
                    "workspace_id": target["workspace_id"],
                    "model_id": target["model_id"],
                    "principal_id": principal_id,
                    "principal_name": clean_string(pick(record, ["User Name", "Name"]), 1000),
                    "principal_type": clean_string(pick(record, ["User Type", "Principal Type"]), 200),
                    "principal_upn": principal_upn.lower() if principal_upn else None,
                    "permissions": clean_string(pick(record, ["Permissions", "Dataset User Access Right"]), 2000),
                    "additional_permissions": clean_string(pick(record, ["Additional Permissions"]), 2000),
                    "raw_json": json_dumps(raw_row.to_dict()),
                    "captured_at": utcnow(),
                })
            statuses["access_snapshot"] = "SUCCEEDED"
        except Exception as exc:
            statuses["access_snapshot"] = "FAILED"
            errors["access_snapshot"] = truncate_error(exc)

    core = [statuses["bpa"] if run_bpa else "NOT_RUN", statuses["vpa"] if run_vertipaq else "NOT_RUN"]
    if all(status == "FAILED" for status in core if status != "NOT_RUN"):
        overall = "FAILED"
    elif any(status == "FAILED" for status in statuses.values()):
        overall = "PARTIAL"
    else:
        overall = "SUCCEEDED"

    completed = utcnow()
    model_row = {
        "scan_id": SCAN_ID,
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_id": target["model_id"],
        "model_name": target["model_name"],
        "scope_source": target["scope_source"],
        "scanner_version": SCANNER_VERSION,
        "solution_stage": SOLUTION_STAGE,
        "permission_precheck_status": target.get("permission_precheck_status"),
        "scanner_workspace_role": target.get("scanner_workspace_role"),
        "permission_precheck_message": target.get("permission_precheck_message"),
        "capacity_id": capacity_id,
        "capacity_name": capacity_name,
        "storage_mode": storage_mode,
        "model_size_bytes": model_size_bytes,
        "overall_status": overall,
        "bpa_status": statuses["bpa"],
        "vpa_status": statuses["vpa"],
        "refresh_status": statuses["refresh"],
        "usage_status": statuses["usage"],
        "direct_lake_status": statuses["direct_lake"],
        "access_snapshot_status": statuses["access_snapshot"],
        "finding_count": len(findings),
        "started_at": started,
        "completed_at": completed,
        "duration_seconds": (completed - started).total_seconds(),
        "error_json": json_dumps(errors) if errors else None,
    }
    dim_model_row = {
        "model_id": target["model_id"],
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_name": target["model_name"],
        "capacity_id": capacity_id,
        "capacity_name": capacity_name,
        "storage_mode": storage_mode,
        "latest_scan_id": SCAN_ID,
        "last_scan_status": overall,
        "last_scanned_at": completed,
    }
    return {
        "model_row": model_row,
        "dim_model_row": dim_model_row,
        "findings": findings,
        "vpa_columns": vpa_columns,
        "vpa_tables": vpa_tables,
        "usage_rows": usage_rows,
        "refresh_rows": refresh_rows,
        "direct_lake_rows": direct_lake_rows,
        "access_rows": access_rows,
    }


In [ ]:
# ---------- Explicit RLS grant synchronization ----------

def sync_explicit_model_access(targets):
    if model_access_sync_mode == "none":
        return
    eligible_targets = [
        target
        for target in targets
        if target.get("permission_precheck_status")
        in {"PASSED", "NOT_CHECKED_USER_MODE", "NOT_ENFORCED"}
    ]
    now = utcnow()
    requested_by = requested_by_upn.strip().lower() or None

    if model_access_sync_mode == "replace_scanner_managed":
        access_table = table_name("model_access")
        for target in eligible_targets:
            current_upns = target["authorized_viewer_upns"]
            predicate = (
                f"model_id = '{target['model_id']}' AND source = 'SCAN_REQUEST_EXPLICIT' AND is_active = true"
            )
            if current_upns:
                escaped = ",".join("'" + upn.replace("'", "''") + "'" for upn in current_upns)
                predicate += f" AND principal_upn NOT IN ({escaped})"
            DeltaTable.forName(spark, access_table).update(
                condition=predicate,
                set={"is_active": "false", "valid_to": "current_timestamp()", "updated_at": "current_timestamp()"},
            )

    rows = []
    for target in eligible_targets:
        for upn in target["authorized_viewer_upns"]:
            rows.append({
                "access_key": stable_id(target["model_id"], upn, "SCAN_REQUEST_EXPLICIT"),
                "workspace_id": target["workspace_id"],
                "model_id": target["model_id"],
                "principal_upn": upn,
                "source": "SCAN_REQUEST_EXPLICIT",
                "is_active": True,
                "valid_from": now,
                "valid_to": None,
                "updated_by": requested_by,
                "updated_at": now,
            })
    upsert_rows("model_access", rows)


In [ ]:
# ---------- V2 AI-friendly current-state consumption contract ----------

"""Deterministic quality grading for scanner findings and recommendations.

This dependency-free source is embedded into the Fabric scanner notebook by
``upgrade_notebook_v2.py`` and is also importable for local contract tests.
"""

ACTIONABLE = "ACTIONABLE"
REVIEW_REQUIRED = "REVIEW_REQUIRED"
INFORMATIONAL = "INFORMATIONAL"
SUPPRESSED = "SUPPRESSED"


def _normalized(value):
    return str(value or "").strip().upper()


def is_system_generated_date_object(finding):
    """Return True for Power BI Auto Date/Time implementation objects."""
    prefixes = ("LOCALDATETABLE_", "DATETABLETEMPLATE_")
    return any(
        _normalized(name).startswith(prefixes)
        for name in (finding.get("table_name"), finding.get("object_name"))
    )


def priority_band(score):
    """Map a stable 0-100 score to an explicit operational priority band."""
    if score >= 80:
        return "P1_CRITICAL"
    if score >= 65:
        return "P2_HIGH"
    if score >= 40:
        return "P3_MEDIUM"
    return "P4_LOW"


def grade_finding(finding):
    """Grade one raw finding without discarding its original evidence."""
    description = str(finding.get("finding_text") or "").strip()
    evidence = str(finding.get("technical_evidence") or "").strip()
    action = str(finding.get("recommended_action") or "").strip()
    severity = _normalized(finding.get("severity"))
    confidence = _normalized(finding.get("confidence"))
    risk = _normalized(finding.get("change_risk"))

    suppression_reason = None
    if not description and not evidence:
        status = SUPPRESSED
        reason = "No description or technical evidence was supplied; retain for audit but exclude from the action queue."
        suppression_reason = reason
    elif is_system_generated_date_object(finding):
        status = SUPPRESSED
        reason = "System-generated Auto Date/Time object; remediate the model-level root cause instead of editing the generated object."
        suppression_reason = reason
    elif not action:
        status = INFORMATIONAL
        reason = "Evidence is retained, but the scanner did not supply a concrete remediation action."
    elif severity in {"INFO", "LOW"}:
        status = INFORMATIONAL
        reason = "Low-severity evidence is useful context but does not belong in the immediate action queue."
    elif confidence in {"LOW", "UNKNOWN"}:
        status = REVIEW_REQUIRED
        reason = "The finding requires human confirmation because confidence is low or unavailable."
    elif risk == "HIGH":
        status = REVIEW_REQUIRED
        reason = "The proposed change has high implementation risk and requires design review before execution."
    else:
        status = ACTIONABLE
        reason = "The finding has evidence, a concrete action, sufficient confidence, and acceptable change risk."

    severity_points = {
        "CRITICAL": 60, "ERROR": 60, "HIGH": 55, "WARNING": 40,
        "MEDIUM": 35, "INFO": 15, "LOW": 5,
    }.get(severity, 0)
    # Existing BPA evidence does not publish confidence. Treat absence as
    # neutral/medium rather than silently emptying the actionable queue.
    confidence_points = {"HIGH": 15, "MEDIUM": 8, "LOW": 0, "UNKNOWN": 0}.get(confidence, 8)
    risk_points = {"LOW": 10, "MEDIUM": 5, "HIGH": -5}.get(risk, 0)
    saving = max(
        int(finding.get("estimated_saving_bytes_low") or 0),
        int(finding.get("estimated_saving_bytes_high") or 0),
    )
    score = severity_points + confidence_points + risk_points
    score += 10 if action else 0
    score += 10 if saving > 0 else 0
    if status == SUPPRESSED:
        score = 0
    elif status == INFORMATIONAL:
        score = min(score, 39)
    score = max(0, min(100, score))

    return {
        "actionability_status": status,
        "actionability_reason": reason,
        "suppression_reason": suppression_reason,
        "finding_priority_score": score,
        "finding_priority_band": priority_band(score),
    }


def _why_it_matters(domain):
    text = _normalized(domain)
    if "DAX" in text or "EXPRESSION" in text:
        return "Improves calculation correctness, maintainability, and representative query performance."
    if any(token in text for token in ("PERFORMANCE", "STORAGE", "VERTIPAQ")):
        return "Reduces model size, refresh cost, memory pressure, and interactive query latency."
    if "FORMAT" in text:
        return "Improves semantic consistency and makes the model easier for users and AI agents to interpret."
    if any(token in text for token in ("MAINTENANCE", "GOVERNANCE", "BEST PRACTICE")):
        return "Reduces support cost and makes future model changes safer and easier to review."
    return "Addresses model quality or operational risk while preserving traceable evidence for validation."


def _validation_method(domain):
    text = _normalized(domain)
    if "DAX" in text or "EXPRESSION" in text:
        return "Compare representative query results and duration before and after the change, then rerun BPA."
    if any(token in text for token in ("PERFORMANCE", "STORAGE", "VERTIPAQ")):
        return "Compare model size, refresh duration, and representative query duration; rerun storage analysis."
    return "Rerun BPA and the scanner, then complete model refresh and report smoke tests."


def grade_recommendation(findings, domain, title, action):
    """Aggregate finding grades into an implementation-oriented recommendation."""
    grades = [grade_finding(row) for row in findings]
    statuses = [grade["actionability_status"] for grade in grades]
    all_auto_date = bool(findings) and all(is_system_generated_date_object(row) for row in findings)

    if all_auto_date:
        status = REVIEW_REQUIRED
        reason = "Generated local date objects were consolidated into one model-level remediation that requires relationship and calculation review."
        title = "Replace Auto Date/Time with an explicit date dimension"
        action = (
            "Disable Auto Date/Time, create and mark an explicit date dimension, update relationships and calculations, "
            "then refresh and regression-test dependent reports."
        )
        score = 72
    else:
        if ACTIONABLE in statuses:
            status = ACTIONABLE
            reason = "At least one linked finding meets the evidence, confidence, action, and risk thresholds for execution."
        elif REVIEW_REQUIRED in statuses:
            status = REVIEW_REQUIRED
            reason = "Linked findings require human confirmation or design review before implementation."
        elif INFORMATIONAL in statuses:
            status = INFORMATIONAL
            reason = "Linked findings are valid context but do not yet form an executable change."
        else:
            status = SUPPRESSED
            reason = "All linked findings are suppressed from the action queue while remaining available for audit."
        score = max((grade["finding_priority_score"] for grade in grades), default=0)
        score += min(statuses.count(ACTIONABLE) + statuses.count(REVIEW_REQUIRED), 10)
        score = max(0, min(100, score))

    highest_risk = max(
        (_normalized(row.get("change_risk")) for row in findings),
        key=lambda value: {"HIGH": 3, "MEDIUM": 2, "LOW": 1}.get(value, 0),
        default="",
    )
    if highest_risk == "HIGH":
        automation = "MANUAL_ONLY"
    elif "FORMAT" in _normalized(domain) and highest_risk in {"LOW", "MEDIUM"}:
        automation = "SCRIPT_CANDIDATE"
    else:
        automation = "MANUAL_REVIEW"

    return {
        "recommendation_title": title,
        "recommended_action": action,
        "actionability_status": status,
        "actionability_reason": reason,
        "recommendation_priority_score": score,
        "recommendation_priority_band": priority_band(score),
        "automation_eligibility": automation,
        "why_it_matters": _why_it_matters(domain),
        "validation_method": _validation_method(domain),
        "rollback_guidance": "Capture the original PBIP/TMDL state in source control and restore it if validation thresholds fail.",
        "actionable_finding_count": statuses.count(ACTIONABLE),
        "suppressed_finding_count": statuses.count(SUPPRESSED),
    }


def grade_opportunity(findings):
    """Aggregate actionability and priority for an opportunity summary."""
    grades = [grade_finding(row) for row in findings]
    statuses = [grade["actionability_status"] for grade in grades]
    all_auto_date = bool(findings) and all(is_system_generated_date_object(row) for row in findings)
    if all_auto_date:
        status, score = REVIEW_REQUIRED, 72
    elif ACTIONABLE in statuses:
        status, score = ACTIONABLE, max(grade["finding_priority_score"] for grade in grades)
    elif REVIEW_REQUIRED in statuses:
        status, score = REVIEW_REQUIRED, max(grade["finding_priority_score"] for grade in grades)
    elif INFORMATIONAL in statuses:
        status, score = INFORMATIONAL, max(grade["finding_priority_score"] for grade in grades)
    else:
        status, score = SUPPRESSED, 0
    score = max(0, min(100, score + min(statuses.count(ACTIONABLE), 10)))
    return {
        "actionability_status": status,
        "actionable_finding_count": statuses.count(ACTIONABLE),
        "review_required_finding_count": statuses.count(REVIEW_REQUIRED),
        "suppressed_finding_count": statuses.count(SUPPRESSED),
        "priority_score": score,
        "priority_band": priority_band(score),
    }


BUSINESS_SCHEMAS = (
    "analysis_control",
    "semantic_model_metadata",
    "semantic_model_vertipaq",
    "semantic_model_best_practice",
    "semantic_model_optimization",
)

OPTIMIZATION_OVERVIEW_SCHEMA = T.StructType([
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("storage_mode", T.StringType()),
    T.StructField("analysis_status", T.StringType()),
    T.StructField("analysis_completed_at", T.TimestampType()),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("semantic_model_size_bytes", T.LongType()),
    T.StructField("optimization_opportunity_count", T.IntegerType()),
    T.StructField("optimization_recommendation_count", T.IntegerType()),
    T.StructField("optimization_finding_count", T.IntegerType()),
    T.StructField("high_severity_finding_count", T.IntegerType()),
    T.StructField("actionable_recommendation_count", T.IntegerType()),
    T.StructField("review_required_recommendation_count", T.IntegerType()),
    T.StructField("suppressed_finding_count", T.IntegerType()),
    T.StructField("best_practice_analysis_status", T.StringType()),
    T.StructField("storage_analysis_status", T.StringType()),
    T.StructField("refresh_history_status", T.StringType()),
    T.StructField("refresh_history_record_count", T.IntegerType()),
    T.StructField("object_usage_analysis_status", T.StringType()),
    T.StructField("object_usage_observation_count", T.IntegerType()),
    T.StructField("direct_lake_analysis_status", T.StringType()),
    T.StructField("direct_lake_observation_count", T.IntegerType()),
    T.StructField("item_access_snapshot_status", T.StringType()),
    T.StructField("item_access_record_count", T.IntegerType()),
    T.StructField("data_availability_explanation", T.StringType()),
])

OPTIMIZATION_OPPORTUNITY_SCHEMA = T.StructType([
    T.StructField("opportunity_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("opportunity_title", T.StringType()),
    T.StructField("optimization_domain", T.StringType()),
    T.StructField("finding_source", T.StringType()),
    T.StructField("highest_severity", T.StringType()),
    T.StructField("finding_count", T.IntegerType()),
    T.StructField("recommendation_count", T.IntegerType()),
    T.StructField("estimated_saving_bytes_low", T.LongType()),
    T.StructField("estimated_saving_bytes_high", T.LongType()),
    T.StructField("change_risk", T.StringType()),
    T.StructField("opportunity_summary", T.StringType()),
    T.StructField("actionability_status", T.StringType()),
    T.StructField("actionable_finding_count", T.IntegerType()),
    T.StructField("review_required_finding_count", T.IntegerType()),
    T.StructField("suppressed_finding_count", T.IntegerType()),
    T.StructField("priority_score", T.IntegerType()),
    T.StructField("priority_band", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

OPTIMIZATION_RECOMMENDATION_SCHEMA = T.StructType([
    T.StructField("recommendation_id", T.StringType(), False),
    T.StructField("opportunity_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("recommendation_title", T.StringType()),
    T.StructField("optimization_domain", T.StringType()),
    T.StructField("recommended_action", T.StringType()),
    T.StructField("change_risk", T.StringType()),
    T.StructField("validation_required", T.BooleanType()),
    T.StructField("estimated_saving_bytes_low", T.LongType()),
    T.StructField("estimated_saving_bytes_high", T.LongType()),
    T.StructField("finding_source", T.StringType()),
    T.StructField("affected_finding_count", T.IntegerType()),
    T.StructField("actionable_finding_count", T.IntegerType()),
    T.StructField("suppressed_finding_count", T.IntegerType()),
    T.StructField("actionability_status", T.StringType()),
    T.StructField("actionability_reason", T.StringType()),
    T.StructField("recommendation_priority_score", T.IntegerType()),
    T.StructField("recommendation_priority_band", T.StringType()),
    T.StructField("automation_eligibility", T.StringType()),
    T.StructField("why_it_matters", T.StringType()),
    T.StructField("validation_method", T.StringType()),
    T.StructField("rollback_guidance", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

OPTIMIZATION_FINDING_SCHEMA = T.StructType([
    T.StructField("finding_id", T.StringType(), False),
    T.StructField("opportunity_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("finding_source", T.StringType()),
    T.StructField("optimization_domain", T.StringType()),
    T.StructField("rule_name", T.StringType()),
    T.StructField("severity", T.StringType()),
    T.StructField("confidence", T.StringType()),
    T.StructField("impact_area", T.StringType()),
    T.StructField("affected_object_type", T.StringType()),
    T.StructField("affected_table_name", T.StringType()),
    T.StructField("affected_object_name", T.StringType()),
    T.StructField("finding_description", T.StringType()),
    T.StructField("recommended_action", T.StringType()),
    T.StructField("technical_evidence", T.StringType()),
    T.StructField("estimated_saving_bytes_low", T.LongType()),
    T.StructField("estimated_saving_bytes_high", T.LongType()),
    T.StructField("change_risk", T.StringType()),
    T.StructField("validation_required", T.BooleanType()),
    T.StructField("actionability_status", T.StringType()),
    T.StructField("actionability_reason", T.StringType()),
    T.StructField("suppression_reason", T.StringType()),
    T.StructField("finding_priority_score", T.IntegerType()),
    T.StructField("finding_priority_band", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

OPTIMIZATION_LINK_SCHEMA = T.StructType([
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("opportunity_id", T.StringType(), False),
    T.StructField("related_entity_id", T.StringType(), False),
])

COLUMN_STORAGE_SCHEMA = T.StructType([
    T.StructField("column_storage_record_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("column_name", T.StringType()),
    T.StructField("data_type", T.StringType()),
    T.StructField("encoding", T.StringType()),
    T.StructField("cardinality", T.LongType()),
    T.StructField("data_size_bytes", T.LongType()),
    T.StructField("dictionary_size_bytes", T.LongType()),
    T.StructField("hierarchy_size_bytes", T.LongType()),
    T.StructField("total_size_bytes", T.LongType()),
    T.StructField("percentage_of_semantic_model_size", T.DoubleType()),
    T.StructField("detected_at", T.TimestampType()),
])

ANALYSIS_RUN_SCHEMA = T.StructType([
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("analysis_profile", T.StringType()),
    T.StructField("analysis_status", T.StringType()),
    T.StructField("permission_precheck_status", T.StringType()),
    T.StructField("best_practice_analysis_status", T.StringType()),
    T.StructField("storage_analysis_status", T.StringType()),
    T.StructField("refresh_history_status", T.StringType()),
    T.StructField("object_usage_analysis_status", T.StringType()),
    T.StructField("direct_lake_analysis_status", T.StringType()),
    T.StructField("item_access_snapshot_status", T.StringType()),
    T.StructField("finding_count", T.IntegerType()),
    T.StructField("started_at", T.TimestampType()),
    T.StructField("completed_at", T.TimestampType()),
    T.StructField("duration_seconds", T.DoubleType()),
    T.StructField("error_details", T.StringType()),
])

SEMANTIC_MODEL_SCHEMA = T.StructType([
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("capacity_id", T.StringType()),
    T.StructField("capacity_name", T.StringType()),
    T.StructField("storage_mode", T.StringType()),
    T.StructField("semantic_model_size_bytes", T.LongType()),
    T.StructField("latest_analysis_id", T.StringType()),
    T.StructField("latest_analysis_status", T.StringType()),
    T.StructField("latest_analysis_at", T.TimestampType()),
    T.StructField("scanner_version", T.StringType()),
])

BEST_PRACTICE_FINDING_SCHEMA = T.StructType([
    T.StructField("best_practice_finding_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("rule_id", T.StringType()),
    T.StructField("rule_name", T.StringType()),
    T.StructField("category", T.StringType()),
    T.StructField("severity", T.StringType()),
    T.StructField("affected_object_type", T.StringType()),
    T.StructField("affected_table_name", T.StringType()),
    T.StructField("affected_object_name", T.StringType()),
    T.StructField("finding_description", T.StringType()),
    T.StructField("recommended_action", T.StringType()),
    T.StructField("technical_evidence", T.StringType()),
    T.StructField("documentation_url", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

TABLE_STORAGE_SCHEMA = T.StructType([
    T.StructField("table_storage_record_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("row_count", T.LongType()),
    T.StructField("data_size_bytes", T.LongType()),
    T.StructField("dictionary_size_bytes", T.LongType()),
    T.StructField("hierarchy_size_bytes", T.LongType()),
    T.StructField("total_size_bytes", T.LongType()),
    T.StructField("percentage_of_semantic_model_size", T.DoubleType()),
    T.StructField("detected_at", T.TimestampType()),
])

CURATED_TABLES = {
    "analysis_runs": ("analysis_control", "semantic_model_analysis_runs", ANALYSIS_RUN_SCHEMA),
    "semantic_models": ("semantic_model_metadata", "semantic_models", SEMANTIC_MODEL_SCHEMA),
    "best_practice_findings": ("semantic_model_best_practice", "semantic_model_best_practice_rule_findings", BEST_PRACTICE_FINDING_SCHEMA),
    "overview": ("semantic_model_optimization", "semantic_model_optimization_overview", OPTIMIZATION_OVERVIEW_SCHEMA),
    "opportunities": ("semantic_model_optimization", "semantic_model_optimization_opportunities", OPTIMIZATION_OPPORTUNITY_SCHEMA),
    "recommendations": ("semantic_model_optimization", "semantic_model_optimization_recommendations", OPTIMIZATION_RECOMMENDATION_SCHEMA),
    "findings": ("semantic_model_optimization", "semantic_model_optimization_findings", OPTIMIZATION_FINDING_SCHEMA),
    "opportunity_recommendation_links": ("semantic_model_optimization", "semantic_model_optimization_opportunity_recommendation_links", OPTIMIZATION_LINK_SCHEMA),
    "opportunity_finding_links": ("semantic_model_optimization", "semantic_model_optimization_opportunity_finding_links", OPTIMIZATION_LINK_SCHEMA),
    "column_storage": ("semantic_model_vertipaq", "semantic_model_column_storage", COLUMN_STORAGE_SCHEMA),
    "table_storage": ("semantic_model_vertipaq", "semantic_model_table_storage", TABLE_STORAGE_SCHEMA),
}

CURRENT_STATE_TABLES = tuple(
    logical_name for logical_name in CURATED_TABLES if logical_name != "analysis_runs"
)


def curated_table_name(logical_name):
    schema_name, physical_name, _ = CURATED_TABLES[logical_name]
    return f"{schema_name}.{physical_name}"


def ensure_curated_tables():
    for schema_name in BUSINESS_SCHEMAS:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema_name}`")
    for logical_name, (_, _, schema) in CURATED_TABLES.items():
        name = curated_table_name(logical_name)
        if not spark.catalog.tableExists(name):
            spark.createDataFrame([], schema).write.format("delta").mode("errorifexists").saveAsTable(name)
            continue
        existing_columns = {field.name.lower() for field in spark.table(name).schema.fields}
        missing_fields = [field for field in schema.fields if field.name.lower() not in existing_columns]
        if missing_fields:
            additions = ", ".join(
                f"`{field.name}` {field.dataType.simpleString().upper()}"
                for field in missing_fields
            )
            spark.sql(f"ALTER TABLE {name} ADD COLUMNS ({additions})")


def replace_semantic_model_current_state(logical_name, semantic_model_id, rows):
    _, _, schema = CURATED_TABLES[logical_name]
    name = curated_table_name(logical_name)
    escaped_model_id = semantic_model_id.replace("'", "''")
    DeltaTable.forName(spark, name).delete(f"semantic_model_id = '{escaped_model_id}'")
    if rows:
        spark.createDataFrame(rows, schema=schema).write.format("delta").mode("append").saveAsTable(name)


def reconcile_workspace_current_state(targets):
    """Remove current-state rows for models no longer eligible in a full workspace scan."""
    workspace_targets = {}
    for target in targets:
        if target.get("scope_source") != "WORKSPACE":
            continue
        workspace_targets.setdefault(target["workspace_id"], set()).add(target["model_id"])

    if not workspace_targets:
        return

    model_dimension = spark.table(curated_table_name("semantic_models"))
    stale_model_ids = set()
    for workspace_id, eligible_model_ids in workspace_targets.items():
        escaped_workspace_id = workspace_id.replace("'", "''")
        existing_model_ids = {
            row["semantic_model_id"]
            for row in (
                model_dimension
                .where(f"workspace_id = '{escaped_workspace_id}'")
                .select("semantic_model_id")
                .collect()
            )
        }
        stale_model_ids.update(existing_model_ids - eligible_model_ids)

    if not stale_model_ids:
        return

    quoted_ids = ", ".join(
        "'" + model_id.replace("'", "''") + "'"
        for model_id in sorted(stale_model_ids)
    )
    predicate = f"semantic_model_id IN ({quoted_ids})"
    for logical_name in CURRENT_STATE_TABLES:
        DeltaTable.forName(spark, curated_table_name(logical_name)).delete(predicate)
    print(
        f"Removed stale current-state rows for {len(stale_model_ids)} semantic model(s) "
        "outside the eligible full-workspace scan scope."
    )


def upsert_curated_history(logical_name, rows, keys):
    if not rows:
        return
    _, _, schema = CURATED_TABLES[logical_name]
    source = spark.createDataFrame(rows, schema=schema)
    condition = " AND ".join(f"t.`{key}` = s.`{key}`" for key in keys)
    (
        DeltaTable.forName(spark, curated_table_name(logical_name))
        .alias("t")
        .merge(source.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


def severity_value(value):
    return {
        "CRITICAL": 5,
        "ERROR": 4,
        "HIGH": 4,
        "WARNING": 3,
        "MEDIUM": 3,
        "INFO": 2,
        "LOW": 1,
    }.get((value or "").upper(), 0)


def risk_value(value):
    return {"HIGH": 3, "MEDIUM": 2, "LOW": 1}.get((value or "").upper(), 0)


def availability_explanations(model_row, result):
    notes = []
    if model_row["refresh_status"] == "SUCCEEDED" and not result["refresh_rows"]:
        notes.append("Refresh history: no records were returned for the selected history window.")
    elif model_row["refresh_status"] == "NOT_RUN":
        notes.append("Refresh history: not requested by this analysis profile.")
    if model_row["usage_status"] == "NOT_RUN":
        notes.append("Object usage: not run in the standard profile; use the deep profile to collect usage evidence.")
    elif model_row["usage_status"] == "SUCCEEDED" and not result["usage_rows"]:
        notes.append("Object usage: analysis completed and returned no observations.")
    if model_row["direct_lake_status"] == "NOT_APPLICABLE":
        notes.append("Direct Lake checks: not applicable because this semantic model uses Import storage.")
    elif model_row["direct_lake_status"] == "SUCCEEDED" and not result["direct_lake_rows"]:
        notes.append("Direct Lake checks: completed with no fallback observations.")
    if model_row["access_snapshot_status"] == "SUCCEEDED" and not result["access_rows"]:
        notes.append("Item access snapshot: completed and returned no explicit access records.")
    return " ".join(notes) or "All requested evidence sources returned data or an explicit status."


def curate_latest_model_analysis(result):
    model_row = result["model_row"]
    analysis_run_rows = [{
        "analysis_id": model_row["scan_id"],
        "workspace_id": model_row["workspace_id"],
        "workspace_name": model_row["workspace_name"],
        "semantic_model_id": model_row["model_id"],
        "semantic_model_name": model_row["model_name"],
        "scanner_version": model_row["scanner_version"],
        "analysis_profile": analysis_profile,
        "analysis_status": model_row["overall_status"],
        "permission_precheck_status": model_row["permission_precheck_status"],
        "best_practice_analysis_status": model_row["bpa_status"],
        "storage_analysis_status": model_row["vpa_status"],
        "refresh_history_status": model_row["refresh_status"],
        "object_usage_analysis_status": model_row["usage_status"],
        "direct_lake_analysis_status": model_row["direct_lake_status"],
        "item_access_snapshot_status": model_row["access_snapshot_status"],
        "finding_count": model_row["finding_count"],
        "started_at": model_row["started_at"],
        "completed_at": model_row["completed_at"],
        "duration_seconds": model_row["duration_seconds"],
        "error_details": model_row["error_json"],
    }]
    upsert_curated_history("analysis_runs", analysis_run_rows, ["analysis_id", "semantic_model_id"])
    if model_row["overall_status"] not in {"SUCCEEDED", "PARTIAL"}:
        return

    analysis_id = model_row["scan_id"]
    semantic_model_id = model_row["model_id"]
    workspace_name = model_row["workspace_name"]
    semantic_model_name = model_row["model_name"]
    findings = result["findings"]

    opportunity_groups = {}
    recommendation_groups = {}
    finding_rows = []
    finding_links = []
    recommendation_links = set()

    for finding in findings:
        finding_quality = grade_finding(finding)
        domain = finding.get("category") or finding.get("impact_area") or "General optimization"
        source = finding.get("source") or "Unknown source"
        opportunity_id = stable_id(semantic_model_id, "OPPORTUNITY", source, domain)
        opportunity = opportunity_groups.setdefault(opportunity_id, {
            "findings": [],
            "recommendations": set(),
            "domain": domain,
            "source": source,
        })
        opportunity["findings"].append(finding)

        recommendation_id = stable_id(
            semantic_model_id,
            "RECOMMENDATION",
            finding.get("rule_id") or finding.get("rule_name"),
            finding.get("recommended_action"),
        )
        recommendation = recommendation_groups.setdefault(recommendation_id, {
            "findings": [],
            "opportunity_id": opportunity_id,
            "domain": domain,
            "source": source,
            "title": finding.get("rule_name") or "Optimization recommendation",
            "action": finding.get("recommended_action"),
        })
        recommendation["findings"].append(finding)
        opportunity["recommendations"].add(recommendation_id)
        recommendation_links.add((opportunity_id, recommendation_id))
        finding_links.append({
            "analysis_id": analysis_id,
            "semantic_model_id": semantic_model_id,
            "opportunity_id": opportunity_id,
            "related_entity_id": finding["finding_id"],
        })
        finding_rows.append({
            "finding_id": finding["finding_id"],
            "opportunity_id": opportunity_id,
            "analysis_id": analysis_id,
            "workspace_name": workspace_name,
            "semantic_model_id": semantic_model_id,
            "semantic_model_name": semantic_model_name,
            "finding_source": source,
            "optimization_domain": domain,
            "rule_name": finding.get("rule_name"),
            "severity": finding.get("severity"),
            "confidence": finding.get("confidence"),
            "impact_area": finding.get("impact_area"),
            "affected_object_type": finding.get("object_type"),
            "affected_table_name": finding.get("table_name"),
            "affected_object_name": finding.get("object_name"),
            "finding_description": finding.get("finding_text"),
            "recommended_action": finding.get("recommended_action"),
            "technical_evidence": finding.get("technical_evidence"),
            "estimated_saving_bytes_low": finding.get("estimated_saving_bytes_low"),
            "estimated_saving_bytes_high": finding.get("estimated_saving_bytes_high"),
            "change_risk": finding.get("change_risk"),
            "validation_required": finding.get("validation_required"),
            **finding_quality,
            "detected_at": finding.get("detected_at"),
        })

    opportunity_rows = []
    for opportunity_id, group in opportunity_groups.items():
        grouped_findings = group["findings"]
        highest = max(grouped_findings, key=lambda row: severity_value(row.get("severity")))
        highest_risk = max(grouped_findings, key=lambda row: risk_value(row.get("change_risk")))
        opportunity_quality = grade_opportunity(grouped_findings)
        opportunity_rows.append({
            "opportunity_id": opportunity_id,
            "analysis_id": analysis_id,
            "workspace_name": workspace_name,
            "semantic_model_id": semantic_model_id,
            "semantic_model_name": semantic_model_name,
            "opportunity_title": f"{group['domain']} optimization",
            "optimization_domain": group["domain"],
            "finding_source": group["source"],
            "highest_severity": highest.get("severity"),
            "finding_count": len(grouped_findings),
            "recommendation_count": len(group["recommendations"]),
            "estimated_saving_bytes_low": sum(row.get("estimated_saving_bytes_low") or 0 for row in grouped_findings),
            "estimated_saving_bytes_high": sum(row.get("estimated_saving_bytes_high") or 0 for row in grouped_findings),
            "change_risk": highest_risk.get("change_risk"),
            "opportunity_summary": f"{len(grouped_findings)} finding(s) from {group['source']} require review in {group['domain']}.",
            **opportunity_quality,
            "detected_at": max(row.get("detected_at") for row in grouped_findings if row.get("detected_at")),
        })

    recommendation_rows = []
    for recommendation_id, group in recommendation_groups.items():
        grouped_findings = group["findings"]
        highest_risk = max(grouped_findings, key=lambda row: risk_value(row.get("change_risk")))
        recommendation_quality = grade_recommendation(
            grouped_findings, group["domain"], group["title"], group["action"]
        )
        recommendation_title = recommendation_quality.pop("recommendation_title")
        recommended_action = recommendation_quality.pop("recommended_action")
        recommendation_rows.append({
            "recommendation_id": recommendation_id,
            "opportunity_id": group["opportunity_id"],
            "analysis_id": analysis_id,
            "workspace_name": workspace_name,
            "semantic_model_id": semantic_model_id,
            "semantic_model_name": semantic_model_name,
            "recommendation_title": recommendation_title,
            "optimization_domain": group["domain"],
            "recommended_action": recommended_action,
            "change_risk": highest_risk.get("change_risk"),
            "validation_required": any(row.get("validation_required") for row in grouped_findings),
            "estimated_saving_bytes_low": sum(row.get("estimated_saving_bytes_low") or 0 for row in grouped_findings),
            "estimated_saving_bytes_high": sum(row.get("estimated_saving_bytes_high") or 0 for row in grouped_findings),
            "finding_source": group["source"],
            "affected_finding_count": len(grouped_findings),
            **recommendation_quality,
            "detected_at": max(row.get("detected_at") for row in grouped_findings if row.get("detected_at")),
        })

    recommendation_link_rows = [
        {
            "analysis_id": analysis_id,
            "semantic_model_id": semantic_model_id,
            "opportunity_id": opportunity_id,
            "related_entity_id": recommendation_id,
        }
        for opportunity_id, recommendation_id in sorted(recommendation_links)
    ]

    column_storage_rows = [{
        "column_storage_record_id": row["evidence_id"],
        "analysis_id": analysis_id,
        "workspace_name": workspace_name,
        "semantic_model_id": semantic_model_id,
        "semantic_model_name": semantic_model_name,
        "table_name": row.get("table_name"),
        "column_name": row.get("column_name"),
        "data_type": row.get("data_type"),
        "encoding": row.get("encoding"),
        "cardinality": row.get("cardinality"),
        "data_size_bytes": row.get("data_size_bytes"),
        "dictionary_size_bytes": row.get("dictionary_size_bytes"),
        "hierarchy_size_bytes": row.get("hierarchy_size_bytes"),
        "total_size_bytes": row.get("total_size_bytes"),
        "percentage_of_semantic_model_size": row.get("model_size_pct"),
        "detected_at": row.get("detected_at"),
    } for row in result["vpa_columns"]]

    table_storage_rows = [{
        "table_storage_record_id": row["evidence_id"],
        "analysis_id": analysis_id,
        "workspace_name": workspace_name,
        "semantic_model_id": semantic_model_id,
        "semantic_model_name": semantic_model_name,
        "table_name": row.get("table_name"),
        "row_count": row.get("row_count"),
        "data_size_bytes": row.get("data_size_bytes"),
        "dictionary_size_bytes": row.get("dictionary_size_bytes"),
        "hierarchy_size_bytes": row.get("hierarchy_size_bytes"),
        "total_size_bytes": row.get("total_size_bytes"),
        "percentage_of_semantic_model_size": row.get("model_size_pct"),
        "detected_at": row.get("detected_at"),
    } for row in result["vpa_tables"]]

    best_practice_rows = [{
        "best_practice_finding_id": finding["finding_id"],
        "analysis_id": analysis_id,
        "workspace_name": workspace_name,
        "semantic_model_id": semantic_model_id,
        "semantic_model_name": semantic_model_name,
        "rule_id": finding.get("rule_id"),
        "rule_name": finding.get("rule_name"),
        "category": finding.get("category"),
        "severity": finding.get("severity"),
        "affected_object_type": finding.get("object_type"),
        "affected_table_name": finding.get("table_name"),
        "affected_object_name": finding.get("object_name"),
        "finding_description": finding.get("finding_text"),
        "recommended_action": finding.get("recommended_action"),
        "technical_evidence": finding.get("technical_evidence"),
        "documentation_url": finding.get("documentation_url"),
        "detected_at": finding.get("detected_at"),
    } for finding in findings if (finding.get("source") or "").upper() == "BPA"]

    semantic_model_rows = [{
        "semantic_model_id": semantic_model_id,
        "workspace_id": model_row["workspace_id"],
        "workspace_name": workspace_name,
        "semantic_model_name": semantic_model_name,
        "capacity_id": model_row["capacity_id"],
        "capacity_name": model_row["capacity_name"],
        "storage_mode": model_row["storage_mode"],
        "semantic_model_size_bytes": model_row["model_size_bytes"],
        "latest_analysis_id": analysis_id,
        "latest_analysis_status": model_row["overall_status"],
        "latest_analysis_at": model_row["completed_at"],
        "scanner_version": model_row["scanner_version"],
    }]

    overview_rows = [{
        "analysis_id": analysis_id,
        "workspace_id": model_row["workspace_id"],
        "workspace_name": workspace_name,
        "semantic_model_id": semantic_model_id,
        "semantic_model_name": semantic_model_name,
        "storage_mode": model_row["storage_mode"],
        "analysis_status": model_row["overall_status"],
        "analysis_completed_at": model_row["completed_at"],
        "scanner_version": model_row["scanner_version"],
        "semantic_model_size_bytes": model_row["model_size_bytes"],
        "optimization_opportunity_count": len(opportunity_rows),
        "optimization_recommendation_count": len(recommendation_rows),
        "optimization_finding_count": len(finding_rows),
        "high_severity_finding_count": sum((row.get("severity") or "").upper() in {"HIGH", "CRITICAL", "ERROR"} for row in findings),
        "actionable_recommendation_count": sum(row["actionability_status"] == ACTIONABLE for row in recommendation_rows),
        "review_required_recommendation_count": sum(row["actionability_status"] == REVIEW_REQUIRED for row in recommendation_rows),
        "suppressed_finding_count": sum(row["actionability_status"] == SUPPRESSED for row in finding_rows),
        "best_practice_analysis_status": model_row["bpa_status"],
        "storage_analysis_status": model_row["vpa_status"],
        "refresh_history_status": model_row["refresh_status"],
        "refresh_history_record_count": len(result["refresh_rows"]),
        "object_usage_analysis_status": model_row["usage_status"],
        "object_usage_observation_count": len(result["usage_rows"]),
        "direct_lake_analysis_status": model_row["direct_lake_status"],
        "direct_lake_observation_count": len(result["direct_lake_rows"]),
        "item_access_snapshot_status": model_row["access_snapshot_status"],
        "item_access_record_count": len(result["access_rows"]),
        "data_availability_explanation": availability_explanations(model_row, result),
    }]

    replacements = {
        "semantic_models": semantic_model_rows,
        "best_practice_findings": best_practice_rows,
        "overview": overview_rows,
        "opportunities": opportunity_rows,
        "recommendations": recommendation_rows,
        "findings": finding_rows,
        "opportunity_recommendation_links": recommendation_link_rows,
        "opportunity_finding_links": finding_links,
        "column_storage": column_storage_rows,
        "table_storage": table_storage_rows,
    }
    for logical_name, rows in replacements.items():
        replace_semantic_model_current_state(logical_name, semantic_model_id, rows)


In [ ]:
# ---------- Execute scan and persist each model immediately ----------

run_error = None
model_results = []
targets = []

try:
    ensure_tables()
    ensure_curated_tables()
    if initialize_only:
        init_summary = {"status": "INITIALIZED", "raw_table_count": len(TABLES), "curated_table_count": len(CURATED_TABLES)}
        print(json.dumps(init_summary, indent=2))
    else:
        with authentication_context():
            targets = resolve_targets()
            print(
                f"Resolved {len(targets)} semantic model(s) across "
                f"{len({target['workspace_id'] for target in targets})} workspace(s)."
            )
            display(pd.DataFrame([
                {
                    "Workspace": target["workspace_name"],
                    "Semantic model": target["model_name"],
                    "Permission check": target["permission_precheck_status"],
                }
                for target in targets
            ]))
    
            run_row = {
                "scan_id": SCAN_ID,
                "request_id": REQUEST_ID,
                "requested_by_upn": requested_by_upn.strip().lower() or None,
                "auth_mode": auth_mode,
                "analysis_profile": analysis_profile,
                "scanner_version": SCANNER_VERSION,
                "solution_stage": SOLUTION_STAGE,
                "status": "RUNNING",
                "started_at": RUN_STARTED_AT,
                "completed_at": None,
                "target_count": len(targets),
                "success_count": 0,
                "partial_count": 0,
                "failed_count": 0,
                "skipped_count": 0,
                "semantic_link_version": SEMANTIC_LINK_VERSION,
                "semantic_link_labs_version": SEMANTIC_LINK_LABS_VERSION,
                "parameters_hash": safe_parameters_hash(),
                "error_category": None,
                "error_message": None,
            }
            upsert_rows("scan_run", [run_row])
    
            sync_explicit_model_access(targets)
    
            for target in targets:
                result = scan_one_model(target)
                model_results.append(result["model_row"])
                upsert_rows("model_scan", [result["model_row"]])
                upsert_rows("dim_model", [result["dim_model_row"]])
                upsert_rows("finding", result["findings"])
                upsert_rows("vpa_column", result["vpa_columns"])
                upsert_rows("vpa_table", result["vpa_tables"])
                upsert_rows("object_usage", result["usage_rows"])
                upsert_rows("refresh", result["refresh_rows"])
                upsert_rows("direct_lake", result["direct_lake_rows"])
                upsert_rows("item_access_snapshot", result["access_rows"])
                curate_latest_model_analysis(result)
            reconcile_workspace_current_state(targets)

except Exception as exc:
    run_error = exc
    print(traceback.format_exc())

completed_at = utcnow()
success_count = sum(row["overall_status"] == "SUCCEEDED" for row in model_results)
partial_count = sum(row["overall_status"] == "PARTIAL" for row in model_results)
failed_count = sum(row["overall_status"] == "FAILED" for row in model_results)
skipped_count = sum(row["overall_status"] == "SKIPPED_PERMISSION" for row in model_results)

if initialize_only:
    final_status = "INITIALIZED"
elif run_error is not None:
    final_status = "FAILED"
elif failed_count and fail_pipeline_if_any_model_fails:
    final_status = "FAILED"
elif skipped_count and fail_pipeline_if_permission_precheck_fails:
    final_status = "FAILED"
elif model_results and skipped_count == len(model_results):
    final_status = "SKIPPED_PERMISSION"
elif failed_count or partial_count or skipped_count:
    final_status = "PARTIAL"
else:
    final_status = "SUCCEEDED"

final_run_row = {
    "scan_id": SCAN_ID,
    "request_id": REQUEST_ID,
    "requested_by_upn": requested_by_upn.strip().lower() or None,
    "auth_mode": auth_mode,
    "analysis_profile": analysis_profile,
    "scanner_version": SCANNER_VERSION,
    "solution_stage": SOLUTION_STAGE,
    "status": final_status,
    "started_at": RUN_STARTED_AT,
    "completed_at": completed_at,
    "target_count": len(targets),
    "success_count": success_count,
    "partial_count": partial_count,
    "failed_count": failed_count,
    "skipped_count": skipped_count,
    "semantic_link_version": SEMANTIC_LINK_VERSION,
    "semantic_link_labs_version": SEMANTIC_LINK_LABS_VERSION,
    "parameters_hash": safe_parameters_hash(),
    "error_category": (
        error_category(run_error)
        if run_error
        else "AUTHORIZATION"
        if skipped_count and fail_pipeline_if_permission_precheck_fails
        else None
    ),
    "error_message": (
        truncate_error(run_error)
        if run_error
        else f"{skipped_count} model(s) skipped because the SPN workspace access precheck did not pass."
        if skipped_count and fail_pipeline_if_permission_precheck_fails
        else None
    ),
}

# If initialization failed before table creation, this write may also fail; keep the original error visible.
try:
    if not initialize_only and spark.catalog.tableExists(table_name("scan_run")):
        upsert_rows("scan_run", [final_run_row])
except Exception as persistence_exc:
    print(f"Unable to persist final run status: {truncate_error(persistence_exc)}")

summary = {
    "scan_id": SCAN_ID,
    "request_id": REQUEST_ID,
    "status": final_status,
    "target_count": len(targets),
    "success_count": success_count,
    "partial_count": partial_count,
    "failed_count": failed_count,
    "skipped_count": skipped_count,
    "duration_seconds": round((completed_at - RUN_STARTED_AT).total_seconds(), 2),
    "model_results": [
        {
            "workspace_id": row["workspace_id"],
            "model_id": row["model_id"],
            "model_name": row["model_name"],
            "status": row["overall_status"],
            "permission_precheck_status": row["permission_precheck_status"],
            "scanner_workspace_role": row["scanner_workspace_role"],
            "finding_count": row["finding_count"],
        }
        for row in model_results
    ],
    "error": truncate_error(run_error) if run_error else None,
}
print(
    f"Scan {final_status}: {success_count} succeeded, {partial_count} partial, "
    f"{failed_count} failed, {skipped_count} skipped. Scan ID: {SCAN_ID}"
)
if summary["model_results"]:
    display(pd.DataFrame(summary["model_results"]))
print(json.dumps(summary, indent=2))

if (
    run_error is not None
    or (failed_count and fail_pipeline_if_any_model_fails)
    or (skipped_count and fail_pipeline_if_permission_precheck_fails)
):
    raise RuntimeError(json.dumps(summary)) from run_error

if exit_notebook_with_summary:
    notebookutils.notebook.exit(json.dumps(summary))
